# Exploration séquentielle — lineage et qualité des données

Objectif : comprendre le parcours de chaque enregistrement, qualifier les zéros et valeurs manquantes, repérer les anomalies réellement bloquantes, puis construire des agrégats fiables sans écraser la source.

## Grille STAR du projet

Pour rendre `STAR` opérationnel ici : **S — Sévérité** (impact métier), **T — Traçabilité** (peut-on remonter à la source ?), **A — Action** (règle automatique ou arbitrage métier), **R — Réversibilité** (peut-on restaurer la ligne originale ?). Cette grille est une convention de travail du projet, pas une norme statistique.

Règle d'or : aucune ligne source n'est supprimée. On ajoute un statut, une raison et une décision ; les agrégats utilisent une vue explicitement qualifiée.

In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd
import numpy as np
try:
    from IPython.display import display
except ImportError:
    display = print

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 100)

def locate(name):
    candidates = [Path(name), Path('..') / name, Path('../..') / name]
    for p in candidates:
        if p.exists(): return p.resolve()
    raise FileNotFoundError(f'{name} absent. Déposer les sources à la racine du dépôt ou adapter locate().')

SCHEDULE_PATH = locate('released_schedule.csv')
DISTANCE_PATH = locate('distances.csv')
print('Notebook cwd:', Path.cwd())
print('Schedule:', SCHEDULE_PATH)
print('Distances:', DISTANCE_PATH)

Notebook cwd: j:\delivery_schedule_optimization\docs
Schedule: J:\delivery_schedule_optimization\docs\released_schedule.csv
Distances: J:\delivery_schedule_optimization\docs\distances.csv


## 1. Manifeste des sources — première étape du lineage

On fige nom, taille et SHA-256 avant toute transformation. Une analyse n'est comparable à une autre que si les sources sont identiques ou si leur différence est expliquée.

In [2]:
def manifest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return {'file': path.name, 'path': str(path), 'bytes': path.stat().st_size, 'sha256': h.hexdigest()}

source_manifest = pd.DataFrame([manifest(SCHEDULE_PATH), manifest(DISTANCE_PATH)])
display(source_manifest)

,file,path,bytes,sha256
0,released_schedule.csv,J:\delivery_schedule_optimization\docs\release...,512330,37370e70146362612b22a8c36c8fc2dc6caf4deb9711e7...
1,distances.csv,J:\delivery_schedule_optimization\docs\distanc...,2205,0d6a4144a39bd8d0311f726ca912acb196d58315e54b80...


## 2. Chargement brut et identifiants de lignage

Les colonnes sont d'abord chargées comme texte : on conserve la représentation reçue et on crée un identifiant stable basé sur le fichier et le numéro de ligne. Les colonnes typées seront ajoutées ensuite.

In [3]:
schedule_raw = pd.read_csv(SCHEDULE_PATH, dtype='string', keep_default_na=False)
schedule_raw.insert(0, 'source_file', SCHEDULE_PATH.name)
schedule_raw.insert(1, 'source_row', np.arange(2, len(schedule_raw) + 2))
schedule_raw.insert(2, 'source_id', [f'{SCHEDULE_PATH.name}:{i}' for i in schedule_raw.source_row])
distance_raw = pd.read_csv(DISTANCE_PATH, dtype='string', keep_default_na=False)
distance_raw.insert(0, 'source_file', DISTANCE_PATH.name)
distance_raw.insert(1, 'source_row', np.arange(2, len(distance_raw) + 2))
distance_raw.insert(2, 'source_id', [f'{DISTANCE_PATH.name}:{i}' for i in distance_raw.source_row])
print('Schedule', schedule_raw.shape, '| distances', distance_raw.shape)
display(schedule_raw.head(3)); display(distance_raw.head(3))

Schedule (5728, 13) | distances (156, 5)


,source_file,source_row,source_id,date,supplier_id,material,plant,unloading_point,schedule_quantity,weight,volume,transmission_date,version_date
0,released_schedule.csv,2,released_schedule.csv:2,2026-11-10,SUP_00091,MAT_000110,PLT_004,ULP_0050,0.0,0.0,0.4116,2026-11-06,2026-11-08
1,released_schedule.csv,3,released_schedule.csv:3,2026-11-11,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116,2026-11-06,2026-11-08
2,released_schedule.csv,4,released_schedule.csv:4,2026-11-12,SUP_00091,MAT_000110,PLT_004,ULP_0050,0.0,0.0,0.4116,2026-11-06,2026-11-08


,source_file,source_row,source_id,supplier_id,distance
0,distances.csv,2,distances.csv:2,SUP_00001,47
1,distances.csv,3,distances.csv:3,SUP_00002,
2,distances.csv,4,distances.csv:4,SUP_00003,83


## 3. Profil du schéma et conversions contrôlées

Avant de convertir, on distingue chaînes vides, chaînes non numériques, valeurs non finies et valeurs numériques valides. Les erreurs de conversion restent visibles dans le jeu qualité.

In [4]:
expected_schedule = ['date','supplier_id','material','plant','unloading_point','schedule_quantity','weight','volume','transmission_date','version_date']
expected_distance = ['supplier_id','distance']
schema_report = pd.DataFrame([
 {'dataset':'schedule','missing_columns':sorted(set(expected_schedule)-set(schedule_raw.columns)),'unexpected_columns':sorted(set(schedule_raw.columns)-set(expected_schedule)-{'source_file','source_row','source_id'})},
 {'dataset':'distances','missing_columns':sorted(set(expected_distance)-set(distance_raw.columns)),'unexpected_columns':sorted(set(distance_raw.columns)-set(expected_distance)-{'source_file','source_row','source_id'})}
])
display(schema_report)

schedule = schedule_raw.copy()
issues = []
numeric_cols = ['schedule_quantity','weight','volume']
for col in numeric_cols:
    raw = schedule[col].str.strip()
    parsed = pd.to_numeric(raw.replace('', pd.NA), errors='coerce')
    bad = raw.ne('') & parsed.isna()
    nonfinite = parsed.notna() & ~np.isfinite(parsed.astype('float64'))
    for idx in schedule.index[bad | nonfinite]:
        issues.append({'source_id':schedule.at[idx,'source_id'],'dataset':'schedule','field':col,'raw_value':schedule.at[idx,col], 'issue':'non_numeric_or_nonfinite'})
    schedule[col] = parsed.astype('Float64')
for col in ['date','transmission_date','version_date']:
    raw = schedule[col].str.strip()
    parsed = pd.to_datetime(raw.replace('', pd.NA), errors='coerce')
    bad = raw.ne('') & parsed.isna()
    for idx in schedule.index[bad]:
        issues.append({'source_id':schedule.at[idx,'source_id'],'dataset':'schedule','field':col,'raw_value':schedule.at[idx,col], 'issue':'invalid_date'})
    schedule[col] = parsed
distances = distance_raw.copy()
distances['distance'] = pd.to_numeric(distances.distance.str.strip().replace('', pd.NA), errors='coerce').astype('Float64')
display(schedule[numeric_cols].describe().T)

,dataset,missing_columns,unexpected_columns
0,schedule,[],[]
1,distances,[],[]


,count,mean,std,min,25%,50%,75%,max
schedule_quantity,5728.0,1230.838687,4254.230773,0.0,48.0,300.0,905.5,100000.0
weight,5559.0,358.758566,1402.243483,0.0,11.03,117.54,247.2,58267.3
volume,5581.0,0.656328,0.597206,0.00882,0.0672,0.57,1.1424,5.808


In [5]:
parsed.head(10)

0   2026-11-08
1   2026-11-08
2   2026-11-08
3   2026-11-08
4   2026-11-08
5   2026-11-08
6   2026-11-08
7   2026-11-08
8   2026-11-08
9   2026-11-08
Name: version_date, dtype: datetime64[us]

## 4. Grain, clés et faux positifs de doublon

Une répétition d'un fournisseur/matière/date n'est pas automatiquement un doublon : plusieurs quais ou lignes de programme peuvent être légitimes. On compare les candidats sur une clé métier complète et on distingue répétitions identiques, conflits de charge et multi-quais.

In [ ]:
business_key = ['date','supplier_id','material','plant','unloading_point']
payload = ['schedule_quantity','weight','volume','transmission_date','version_date']
candidate = schedule.groupby(business_key, dropna=False, observed=True)
dup_summary = candidate.agg(rows=('source_id','size'), payload_variants=('source_id', lambda s: schedule.loc[s.index,payload].drop_duplicates().shape[0]), source_ids=('source_id', lambda s: list(s)))
dup_summary = dup_summary[dup_summary.rows.gt(1)].reset_index()
dup_summary['classification'] = np.select([dup_summary.payload_variants.eq(1)], ['exact_duplicate_candidate'], default='same_key_conflicting_payload')
print('Clés métier répétées:', len(dup_summary))
display(dup_summary.head(20))

pair_key = ['supplier_id','plant']
material_key = ['supplier_id','plant','material']
grain_report = pd.DataFrame([
 {'grain':'source rows','count':len(schedule)},
 {'grain':'unique supplier × plant','count':schedule[pair_key].drop_duplicates().shape[0]},
 {'grain':'unique supplier × plant × material','count':schedule[material_key].drop_duplicates().shape[0]},
 {'grain':'unique supplier × plant × date','count':schedule[['supplier_id','plant','date']].drop_duplicates().shape[0]},
 {'grain':'unique source id','count':schedule.source_id.nunique()}
])
display(grain_report)

## 5. Zéros, valeurs manquantes et « couples nuls »

On ne confond pas : quantité nulle (aucun besoin ce jour), poids/volume nul (potentiellement charge invalide), chaîne vide (information absente), et couple fournisseur × usine sans aucun besoin positif. Les couples inactifs restent dans le rapport, jamais dans la consolidation active.

In [ ]:
q = schedule.schedule_quantity
zero_report = pd.DataFrame([
 {'condition':'quantity == 0','rows':int(q.eq(0).fillna(False).sum())},
 {'condition':'quantity > 0','rows':int(q.gt(0).fillna(False).sum())},
 {'condition':'quantity < 0','rows':int(q.lt(0).fillna(False).sum())},
 {'condition':'quantity missing/invalid','rows':int(q.isna().sum())},
 {'condition':'positive quantity with zero weight','rows':int((q.gt(0)&schedule.weight.eq(0)).fillna(False).sum())},
 {'condition':'positive quantity with zero volume','rows':int((q.gt(0)&schedule.volume.eq(0)).fillna(False).sum())},
 {'condition':'positive quantity with missing weight','rows':int((q.gt(0)&schedule.weight.isna()).fillna(False).sum())},
 {'condition':'positive quantity with missing volume','rows':int((q.gt(0)&schedule.volume.isna()).fillna(False).sum())}
])
display(zero_report)

pair_status = schedule.assign(is_active=q.gt(0).fillna(False)).groupby(pair_key, dropna=False).is_active.any().rename('has_positive_need').reset_index()
inactive_pairs = pair_status[~pair_status.has_positive_need]
print('Couples présents:', len(pair_status), '| couples sans besoin positif:', len(inactive_pairs))
display(inactive_pairs)

## 6. Lineage temporel — dates, versions et séquences

On vérifie les bornes, les versions, les transmissions postérieures au besoin et les trous de calendrier par matière/couple. Une transmission postérieure à la date de besoin est critique, mais ne prouve pas à elle seule une erreur : les dates peuvent représenter des versions de programme différentes.

In [ ]:
temporal = schedule.copy()
temporal['transmission_after_need'] = temporal.transmission_date.gt(temporal.date)
temporal['version_after_need'] = temporal.version_date.gt(temporal.date)
temporal['version_before_transmission'] = temporal.version_date.lt(temporal.transmission_date)
temporal_summary = pd.DataFrame([
 {'check':'date range','value':f'{temporal.date.min()} → {temporal.date.max()}'},
 {'check':'transmission after need','value':int(temporal.transmission_after_need.sum())},
 {'check':'version after need','value':int(temporal.version_after_need.sum())},
 {'check':'version before transmission','value':int(temporal.version_before_transmission.sum())},
 {'check':'unique version dates','value':temporal.version_date.nunique()},
 {'check':'unique transmission dates','value':temporal.transmission_date.nunique()}
])
display(temporal_summary)
display(temporal.loc[temporal.transmission_after_need, ['source_id']+business_key+['transmission_date','version_date']].head(20))

## 7. STAR — scorer et prioriser les anomalies

Le score ci-dessous donne la priorité aux erreurs qui peuvent créer un retard, une sous-estimation de capacité ou un coût faux. Les règles sont explicites et modifiables. Une observation sans preuve est classée `review`, pas supprimée.

In [ ]:
issue_rows = list(issues)
for idx, row in schedule.iterrows():
    qv, w, v = row.schedule_quantity, row.weight, row.volume
    common = {'source_id':row.source_id,'dataset':'schedule','field':'','raw_value':'','supplier_id':row.supplier_id,'plant':row.plant,'material':row.material,'date':row.date}
    if pd.notna(qv) and qv < 0:
        issue_rows.append({**common,'field':'schedule_quantity','raw_value':qv,'issue':'negative_quantity','severity':'critical','action':'quarantine'})
    elif pd.notna(qv) and qv > 0 and (pd.isna(w) or pd.isna(v)):
        issue_rows.append({**common,'field':'weight/volume','raw_value':f'{w}/{v}','issue':'positive_need_missing_load','severity':'critical','action':'block_pair_until_resolved'})
    elif pd.notna(qv) and qv > 0 and (w == 0 or v == 0):
        issue_rows.append({**common,'field':'weight/volume','raw_value':f'{w}/{v}','issue':'positive_need_zero_load','severity':'high','action':'review_source'})
    if pd.notna(qv) and qv == 0 and pd.notna(v) and v > 0:
        issue_rows.append({**common,'field':'volume','raw_value':v,'issue':'zero_quantity_nonzero_volume','severity':'info','action':'exclude_from_load_aggregate_keep_as_product_attribute'})

quality = pd.DataFrame(issue_rows)
severity_rank = {'critical':4,'high':3,'medium':2,'low':1,'info':0}
if not quality.empty:
    quality['severity_rank'] = quality.get('severity', pd.Series(index=quality.index,dtype='string')).map(severity_rank).fillna(1)
    quality['star_priority'] = quality['severity_rank'] * 100 + quality.source_id.map(schedule.source_id.value_counts()).fillna(1).clip(upper=99)
    quality = quality.sort_values(['star_priority','source_id'], ascending=[False,True])
display(quality.head(50))
print('STAR counts:', quality.groupby(['issue','severity','action'], dropna=False).size().sort_values(ascending=False).head(30))

## 8. Résolution des distances — conserver la provenance de la jointure

On vérifie unicité, valeurs invalides, distances absentes et fournisseurs du planning sans correspondance. Une jointure plusieurs-à-plusieurs est interdite : elle gonflerait artificiellement les lignes et les sommes.

In [ ]:
distance_duplicates = distances[distances.duplicated('supplier_id', keep=False)].sort_values('supplier_id')
distance_status = distances.assign(distance_status=np.select([distances.distance.isna().to_numpy(), distances.distance.le(0).fillna(False).to_numpy()], ['missing','nonpositive'], default='valid'))
active_suppliers = set(schedule.loc[q.gt(0).fillna(False),'supplier_id'])
distance_suppliers = set(distances.supplier_id)
join_audit = pd.DataFrame([
 {'metric':'distance rows','value':len(distances)},
 {'metric':'duplicate supplier rows','value':len(distance_duplicates)},
 {'metric':'active suppliers without distance row','value':len(active_suppliers-distance_suppliers)},
 {'metric':'active suppliers with missing distance','value':len(active_suppliers & set(distances.loc[distances.distance.isna(),'supplier_id']))}
])
display(join_audit); display(distance_duplicates.head(20))
if distance_duplicates.empty:
    joined = schedule.merge(distances[['supplier_id','distance','source_id']].rename(columns={'source_id':'distance_source_id'}), on='supplier_id', how='left', validate='many_to_one', indicator='distance_join')
    display(joined.distance_join.value_counts(dropna=False))

## 9. Jeu canonique et agrégats sans faux positifs

On construit une vue d'analyse, sans modifier `schedule_raw`. `quantity == 0` est exclue des besoins à transporter, mais gardée dans le lineage. Les lignes à charge manquante sont mises en quarantaine pour les agrégats de capacité. Les lignes positives nulles restent séparées en revue STAR.

L'agrégation de transport ci-dessous ne doit pas être interprétée comme le volume camion validé : le sens de l'unité volume reste une décision métier.

In [ ]:
schedule['quantity_status'] = np.select([q.isna().to_numpy(), q.lt(0).fillna(False).to_numpy(), q.eq(0).fillna(False).to_numpy(), q.gt(0).fillna(False).to_numpy()], ['invalid','negative','no_need','active'], default='invalid')
missing_load = (schedule.weight.isna() | schedule.volume.isna()).to_numpy()
zero_load = (schedule.weight.eq(0).fillna(False) | schedule.volume.eq(0).fillna(False)).to_numpy()
schedule['load_status'] = np.select([missing_load, zero_load], ['missing','zero_review'], default='complete')
schedule['decision'] = np.select([schedule.quantity_status.ne('active').to_numpy(), schedule.load_status.ne('complete').to_numpy()], ['exclude_no_positive_need','quarantine_load'], default='eligible_for_reviewed_aggregate')
schedule['decision_reason'] = np.select([schedule.quantity_status.eq('no_need').to_numpy(), schedule.quantity_status.eq('negative').to_numpy(), schedule.quantity_status.eq('invalid').to_numpy(), schedule.load_status.eq('missing').to_numpy(), schedule.load_status.eq('zero_review').to_numpy()], ['quantity_zero_no_shipment','negative_quantity','invalid_quantity','missing_capacity_data','zero_capacity_data'], default='positive_need_complete_load')

eligible = schedule[schedule.decision.eq('eligible_for_reviewed_aggregate')].copy()
group_cols = ['date','supplier_id','plant']
daily_pair = eligible.groupby(group_cols, dropna=False).agg(
    source_lines=('source_id','count'), materials=('material','nunique'), unloading_points=('unloading_point','nunique'),
    quantity=('schedule_quantity','sum'), weight_kg=('weight','sum'), declared_volume_sum=('volume','sum'), source_ids=('source_id',list)
).reset_index()
pair_weekly = eligible.assign(week=eligible.date.dt.to_period('W-SUN').astype('string')).groupby(['week','supplier_id','plant'], dropna=False).agg(
    source_lines=('source_id','count'), active_days=('date','nunique'), materials=('material','nunique'), quantity=('schedule_quantity','sum'), weight_kg=('weight','sum'), declared_volume_sum=('volume','sum')
).reset_index()
display(daily_pair.head(20)); display(pair_weekly.head(20))
print('Éligibles:',len(eligible),'| quarantinées capacité:',int(schedule.decision.eq('quarantine_load').sum()),'| zéro besoin:',int(schedule.decision.eq('exclude_no_positive_need').sum()))

## 10. Registre STAR, quarantaine réversible et livrables

Les faux positifs de doublon ne sont pas effacés : seuls les doublons strictement identiques sur clé et payload sont candidats à une déduplication, et une déduplication n'est activée qu'après confirmation métier. Les autres répétitions sont des lignes distinctes à comprendre.

Export du lineage complet, du registre qualité, des candidats doublons et des agrégats qualifiés. Ces sorties sont générées dans `results/exploration/` et ne remplacent jamais les sources.

In [ ]:
OUT = Path('results/exploration')
OUT.mkdir(parents=True, exist_ok=True)
lineage = schedule.copy()
lineage.to_csv(OUT/'schedule_lineage.csv', index=False)
distance_raw.to_csv(OUT/'distances_lineage.csv', index=False)
quality.to_csv(OUT/'star_quality_register.csv', index=False)
dup_summary.to_csv(OUT/'duplicate_candidates.csv', index=False)
inactive_pairs.to_csv(OUT/'inactive_supplier_plant_pairs.csv', index=False)
daily_pair.to_csv(OUT/'daily_pair_aggregate_reviewed.csv', index=False)
pair_weekly.to_csv(OUT/'weekly_pair_aggregate_reviewed.csv', index=False)
source_manifest.to_json(OUT/'source_manifest.json', orient='records', indent=2)
print('Écrits :', OUT.resolve())